### Miguel Baños Baladrón
### Miguel Pérez Francos
### Rodrigo Touceda Tapias

# Importaciones

In [1]:
using CSV, DataFrames, Glob, Statistics

# Preparación de los datos (20%)

## 1. Carga y unificación de los datos

In [2]:
base = "Datos Práctica"

# CSV del Investigador A
csv_inv_a = glob("Investigador A/day */*.csv", base)

# CSV del Investigador B
csv_inv_b = glob("Investigador B/*.csv", base)

all_csv = vcat(csv_inv_a, csv_inv_b)

dfs = [CSV.read(file, DataFrame) for file in all_csv]
df_total = vcat(dfs...)

println("Dataset correctamente cargado y unificado.")
println("Número de variables:         ", ncol(df_total))
println("Número de instancias:        ", nrow(df_total))
println("Número de individuos:        ", length(unique(df_total.subject)))
println("Número de clases de salida:  ", length(unique(df_total.Activity)))

Dataset correctamente cargado y unificado.
Número de variables:         563
Número de instancias:        10299
Número de individuos:        30
Número de clases de salida:  6


In [6]:
println("Clases:\n")
for activity in unique(df_total.Activity)
    println(activity)
end

Clases:

STANDING
SITTING
LAYING
WALKING
WALKING_UPSTAIRS
WALKING_DOWNSTAIRS


# 2. Análisis de valores ausentes

In [7]:
# Porcentajes de nulos por variable
nulos_por_variable = DataFrame(
    Variable = names(df_total),
    PorcentajeNulos = [count(ismissing, df_total[!, col]) / nrow(df_total) * 100 for col in names(df_total)]
)

# Porcentaje de nulos en todo el dataset
total_nulos = sum(count(ismissing, df_total[!, col]) for col in names(df_total))
total_valores = nrow(df_total) * ncol(df_total)

porcentaje_total_nulos = (total_nulos / total_valores) * 100

println("Porcentaje total de valores nulos en el dataset: $(porcentaje_total_nulos)%")

Porcentaje total de valores nulos en el dataset: 0.9984242033534787%


# 3. Tratamiento y transformación de datos

In [ ]:
n = nrow(df_total)

res = DataFrame(variable = String[], n_missing = Int[], porcentaje = Float64[])

for col in names(df_total)
    n_miss = count(ismissing, df_total[!, col])
    porc = round((n_miss / n) * 100, digits=2)
    push!(res, (string(col), n_miss, porc))
end

# Ordenar de mayor a menor porcentaje
sort!(res, :porcentaje, rev=true)

# Mostrar solo las 10 primeras
first(res, 10)

Row,variable,n_missing,porcentaje
,String,Int64,Float64
1,tBodyGyroMag-mad(),1033,10.03
2,tBodyGyroMag-iqr(),1033,10.03
3,fBodyAcc-mad()-Y,1032,10.02
4,fBodyAccJerk-mean()-X,1032,10.02
5,tBodyAccJerk-energy()-X,1030,10.0
6,tBodyAccJerk-entropy()-Y,1030,10.0
7,tBodyAccMag-max(),1030,10.0
8,tGravityAccMag-std(),1030,10.0
9,tGravityAccMag-entropy(),1030,10.0


# Como la variable con mayor porcentaje de nulos no presenta un valor muy alto, decidimos imputar todas las variables

In [8]:
# Rellenamos con la mediana los valores nulos porque es menos sensible a outliers
df_imputado = deepcopy(df_total)
gdf = groupby(df_imputado, :subject) # Agrupamos por individuo para que cada dato nulo se rellene con la mediana de los valores de dicho individuo.

for subdf in gdf
    for col in names(subdf)
        if col in (:subject, :activity)
            continue
        end

        # Ignoramos variables no numéricas
        coldata = subdf[!, col]
        if !(eltype(skipmissing(coldata)) <: Number)
            continue
        end

        mediana = median(skipmissing(coldata))
        replace!(coldata, missing => mediana)
    end
end

In [10]:
# DataFrame sin la variable 'subject' para entrenamiento
df_without_subject = select(df_imputado, Not(:subject));